this is the script to sample a motion-corrected image with trained diffusion models

In [1]:
# import
import sys 
sys.path.append('/workspace/Documents')
import os
import torch
import numpy as np
import nibabel as nb
import Diffusion_for_CT_motion.diffusion_models.conditional_diffusion_3D as ddpm_3D
import Diffusion_for_CT_motion.diffusion_models.conditional_EDM_3D as edm
import Diffusion_for_CT_motion.utils.functions_collection as ff
import Diffusion_for_CT_motion.utils.Build_list as Build_list
import Diffusion_for_CT_motion.utils.Generator as Generator

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/Documents/Diffusion_models/denoising_diffusion_pytorch/denoising_diffusion_pytorch/standard_diffusion.py:773: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/workspace/Documents/Diffusion_models/denoising_diffusion_pytorch/denoising_diffusion_pytorch/conditional_diffusion.py:950: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/workspace/Documents/Diffusion_models/denoising_diffusion_pytorch/denoising_diffusion_pytorch/conditional_diffusion_3D.py:865: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `tor

#### step 1: set the trained diffusion model path

In [2]:
trial_name = 'diffusion_model'
epoch = 49
trained_model_filename = os.path.join('/mnt/camca_NAS/diffusion_ct_motion/models/',trial_name, 'models', 'model-'+str(epoch)+'.pt') # replace with your own path
save_folder = os.path.join('/mnt/camca_NAS/diffusion_ct_motion/models', trial_name, 'pred_image'); os.makedirs(save_folder, exist_ok=True)

#### step 2: set the test cohort
here we use one case as example, in this case we only have motion-corrupted data

In [3]:
# define train
build_sheet =  Build_list.Build('patient_list.xlsx')  # this is data path for training data
_,_, x0_list_test, condition_list_test = build_sheet.__build__(batch_list = [1]) 

#### step 3: set some default parameter

In [4]:
# set default, don't change unless necessary
image_size_3D = [256,256,20] 
patch_size = image_size_3D[0] # apply on whole image
slice_range_list = [[0,20],[10,30],[20,40],[30,50]] # do for every 20-slice stack (overlapped), can change to some other numbers

# don't change the following, 
histogram_equalization = True # alreayd set True
# these two are used for histogram equalization
bins = np.load('bins.npy') # provide these two files in the repo
bins_mapped = np.load('bins_mapped.npy')

# for data normalization 
background_cutoff = -1000 
maximum_cutoff = 2000
normalize_factor = 'equation'

# sample steps:
num_sample_steps = 50 # or 50

#### step 4: define the U-Net and the EDM

In [5]:
# main code
model = ddpm_3D.Unet3D(
    init_dim = 64,
    channels = 1, 
    dim_mults = (1, 2, 4, 8),
    flash_attn = False,
    conditional_diffusion = True,
    full_attn = (None, None, False, True),
)

diffusion_model = edm.EDM(
    model,
    image_size = image_size_3D,
    num_sample_steps = num_sample_steps,
    clip_or_not = True,
    clip_range = [-1,1],)

#### step 5: sample

In [7]:
for i in range(0,x0_list_test.shape[0]):
    
    x0_file = x0_list_test[i]
    condition_file = condition_list_test[i]

    patient_id = os.path.basename(os.path.dirname(condition_file))

    print(i,patient_id)

    save_folder_case = os.path.join(save_folder, patient_id); os.makedirs(save_folder_case, exist_ok=True)

    if os.path.isfile(os.path.join(save_folder_case, 'pred.nii.gz')) == 0:
        for slice_range in slice_range_list:
            
            generator = Generator.Dataset_dual_patch(
                np.array([condition_file]), # in this case, we don't have x0 file (the motion-free reference)
                np.array([condition_file]),

                image_size_3D = image_size_3D,
                slice_start = slice_range[0],
                slice_num = slice_range[1] - slice_range[0],

                patch_size = patch_size,
                patch_stride = 1,
                original_patch_num = 1,
                random_sampled_patch_num = 0,
              
                histogram_equalization = histogram_equalization, 
                bins = bins,
                bins_mapped = bins_mapped,
                
                background_cutoff = background_cutoff, 
                maximum_cutoff = maximum_cutoff,
                normalize_factor = normalize_factor,)

            
            # sample:
            sampler = edm.Sampler(
                diffusion_model,
                generator,
                image_size = image_size_3D,
                batch_size = 1)

            save_file_name = os.path.join(save_folder_case, 'pred-slice' + str(slice_range[0]) +'to' + str(slice_range[1])+ '.nii.gz')
            sampler.sample_3D_w_trained_model(trained_model_filename=trained_model_filename, 
                                        motion_image_file =  condition_file, 
                                        save_file = save_file_name,
                                        slice_range = slice_range)
                                        
  
    print('finish sampling')
    
    # annel the slices
    affine = nb.load(os.path.join(save_folder_case, 'pred-slice30to50.nii.gz')).affine
    slice_0_20_image = nb.load(os.path.join(save_folder_case, 'pred-slice0to20.nii.gz')).get_fdata()
    slice_10_30_image = nb.load(os.path.join(save_folder_case, 'pred-slice10to30.nii.gz')).get_fdata()
    slice_20_40_image = nb.load(os.path.join(save_folder_case, 'pred-slice20to40.nii.gz')).get_fdata()
    slice_30_50_image = nb.load(os.path.join(save_folder_case, 'pred-slice30to50.nii.gz')).get_fdata()

    final_image = np.zeros((slice_0_20_image.shape[0], slice_0_20_image.shape[1], 50))
    final_image[:,:,0:15] = slice_0_20_image[:,:,0:15]
    final_image[:,:,15:25] = slice_10_30_image[:,:,5:15]
    final_image[:,:,25:35] = slice_20_40_image[:,:,5:15]
    final_image[:,:,35:50] = slice_30_50_image[:,:,5:20]
    nb.save(nb.Nifti1Image(final_image,affine ), os.path.join(save_folder_case, 'pred.nii.gz'))




0 case_B


/workspace/Documents/Diffusion_for_CT_motion/diffusion_models/conditional_EDM_3D.py:507: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(trained_model_filena

model device:  cuda:0


sampling time step: 100%|██████████| 100/100 [01:47<00:00,  1.08s/it]


(256, 256, 20)
model device:  cuda:0


...

(256, 256, 20)
model device:  cuda:0


sampling time step: 100%|██████████| 100/100 [01:47<00:00,  1.08s/it]


(256, 256, 20)
model device:  cuda:0


sampling time step: 100%|██████████| 100/100 [01:47<00:00,  1.08s/it]


(256, 256, 20)
finish sampling
